# 🎓 LAB513 Student Runbook Notebook

> **Workshop:** AI-Powered FAQ Assistant with Azure SQL, RAG, Foundry Agents, Fabric & MCP  
> **Duration:** ~2.5 hours

Run the workshop end-to-end from this notebook. It mirrors the module flow in [`STUDENT-GUIDE.md`](../STUDENT-GUIDE.md):

0. Prerequisites & setup
1. AI-enhanced querying with Azure SQL
2. Copilot-assisted SQL development
3. RAG implementation
4. Foundry Agent orchestration (MCP)
5. Microsoft Fabric integration
6. SQL MCP server with Data API Builder


## Usage

- Fill in the **Configuration** cell with the values your instructor gave you (SQL server FQDN, OpenAI endpoint, tenant ID, subscription ID). They are saved to a local, git-ignored `student.config.local.json`.
- Set `DO_EXECUTE = True` to actually run commands. Leave it `False` to preview commands first.
- Run the cells **in order** — later modules depend on earlier ones.
- Some steps (Copilot Chat, Foundry portal, Fabric portal) are done in the browser/VS Code UI; those cells contain guidance and pre-checks.


In [2]:
# Shared helpers
import os
import json
import shutil
import struct
import subprocess
from pathlib import Path


def run(cmd: str, check: bool = True):
    """Run a shell command, stream its output, and return the result."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, text=True, capture_output=True,
        encoding="utf-8", errors="replace"
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return result


def get_access_token(resource: str) -> str:
    """Get an Azure AD access token for a resource via the Azure CLI."""
    result = subprocess.run(
        f'az account get-access-token --resource "{resource}" --query accessToken -o tsv',
        shell=True, text=True, capture_output=True, encoding="utf-8", errors="replace"
    )
    if result.returncode != 0:
        raise RuntimeError(f"Could not get token for {resource}. Run 'az login' first.\n{result.stderr}")
    return result.stdout.strip()


In [ ]:
# ===================== Configuration =====================
# Values your instructor provides are kept OUT of git in a local file:
#   student.config.local.json  (created from the sample below on first run)
DO_EXECUTE = True  # set True to actually run commands

workshop_dir = Path.cwd()
if workshop_dir.name != "workshop":
    # Allow running from repo root too
    candidate = workshop_dir / "workshop"
    workshop_dir = candidate if candidate.exists() else workshop_dir
repo_dir = workshop_dir.parent if workshop_dir.name == "workshop" else workshop_dir

_cfg_local = workshop_dir / "student.config.local.json"
_sample = {
    "tenant_id": "506e2450-c16c-43fd-a391-6a014bf4af13",
    "subscription_id": "aae170e4-5b90-4bac-baf4-7aed0ebd6e4e",
    "sql_server": "faq-ai-assistant-q4s3xh537i3xg.database.windows.net",
    "sql_database": "faq-ai-assistant-db",
    "openai_endpoint": "https://ai-q4s3xh537i3xg.openai.azure.com/",
    "chat_deployment": "gpt-5-mini",
    "chat_api_version": "2025-01-01-preview",
    "embedding_deployment": "text-embedding-ada-002",
    "embedding_api_version": "2024-10-21"
}

if not _cfg_local.exists():
    _cfg_local.write_text(json.dumps(_sample, indent=2), encoding="utf-8")
    raise SystemExit(
        f"Created {_cfg_local.name}.\n"
        "\U0001F449 Fill in the values from your instructor, save, then re-run this cell. "
        "(This file is local and should not be committed.)"
    )

_cfg = json.loads(_cfg_local.read_text(encoding="utf-8"))
TENANT_ID = _cfg["tenant_id"]
SUBSCRIPTION_ID = _cfg["subscription_id"]
SQL_SERVER = _cfg["sql_server"]
SQL_DATABASE = _cfg["sql_database"]
OPENAI_ENDPOINT = _cfg["openai_endpoint"].rstrip("/")
CHAT_DEPLOYMENT = _cfg["chat_deployment"]
CHAT_API_VERSION = _cfg["chat_api_version"]
EMBEDDING_DEPLOYMENT = _cfg["embedding_deployment"]
EMBEDDING_API_VERSION = _cfg["embedding_api_version"]

if any(str(v).startswith("<") or "xxxxx" in str(v) for v in _cfg.values()):
    raise SystemExit(f"Fill in real values in {_cfg_local.name} (still contains placeholders).")

print("Configuration loaded from student.config.local.json")
print("repo_dir     :", repo_dir)
print("SQL server   :", SQL_SERVER)
print("SQL database :", SQL_DATABASE)
print("OpenAI       :", OPENAI_ENDPOINT)


Configuration loaded from student.config.local.json
repo_dir     : /Users/scwong/Documents/projects/ms-workshop-build-ai-app/MS-LAB513-Student
SQL server   : faq-ai-assistant-q4s3xh537i3xg.database.windows.net
SQL database : faq-ai-assistant-db
OpenAI       : https://ai-q4s3xh537i3xg.openai.azure.com


---
## Module 0 — Prerequisites & Setup (15 min)

**Objective:** verify tools, accounts, and services are ready.

First-time only, install workshop tools from the repo root:

```powershell
.\workshop\bootstrap.ps1
```


### 0.0) In-notebook bootstrap check (optional, run once)

If `.\workshop\bootstrap.ps1` failed (e.g. `No module named pip`), or you'd rather not
leave the notebook, run the cell below. It installs packages using **this notebook's
kernel interpreter** directly, so it also verifies the kernel is set up correctly.

#### Troubleshooting bootstrap issues
| Symptom | Cause | Fix |
|---|---|---|
| `No module named pip` (repeated) | The Python found on PATH has no `pip`/`ensurepip` (common with minimal/embeddable installs, or a corrupted install) | Run the cell below — it verifies pip with `pip --version`, then repairs it via `ensurepip` if needed. If that also fails, reinstall Python from [python.org](https://python.org) with "pip" checked, then restart the kernel |
| `ensurepip` fails with `WinError 5 Access is denied` writing to `...\Scripts\pip3.exe`, plus `Ignoring invalid distribution ~ip/~ywin32/...` warnings | Shared/admin-owned Python install (e.g. `c:\PythonXYZ`) the current user can't write to, often combined with leftover files from an interrupted pip self-upgrade | The cell below self-heals: it deletes the stale `~`-prefixed leftover folders, then retries `ensurepip --upgrade --user` (installs to the per-user site instead of the protected `Scripts` folder) |
| `pip install failed in the local environment` | Network/proxy/VPN blocking PyPI, or no write permission to site-packages | Re-run with `--user`, check corporate proxy settings, or run VS Code as the same user that owns the Python install |
| Wrong interpreter used | Multiple Pythons installed; `python` on PATH ≠ the notebook kernel | Check `sys.executable` printed below matches **Select Kernel** in the top-right of this notebook |
| Kernel still can't `import pyodbc`/`mcp` after install | Kernel process needs a restart to pick up new packages | Restart the kernel (Command Palette → "Restart Kernel"), then re-run from the top |
| `pyodbc` driver error later in Module 0 | ODBC Driver for SQL Server not installed | Install [ODBC Driver 18](https://aka.ms/downloadmsodbcsql) and restart VS Code |



In [6]:
import sys
import subprocess
import importlib.util
import sysconfig
import shutil
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version   :", sys.version)

# This cell is designed to be run standalone (e.g. right after a fresh kernel
# start), so it doesn't assume the "Shared helpers" or "Configuration" cells
# above have already run. Define what it needs if not already present.
if "repo_dir" not in globals():
    _workshop_dir = Path.cwd()
    if _workshop_dir.name != "workshop":
        _candidate = _workshop_dir / "workshop"
        _workshop_dir = _candidate if _candidate.exists() else _workshop_dir
    repo_dir = _workshop_dir.parent if _workshop_dir.name == "workshop" else _workshop_dir

if "run" not in globals():
    def run(cmd: str, check: bool = True):
        """Run a shell command, stream its output, and return the result."""
        print(f"$ {cmd}")
        result = subprocess.run(
            cmd, shell=True, text=True, capture_output=True,
            encoding="utf-8", errors="replace"
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if check and result.returncode != 0:
            raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
        return result


def _pip_is_functional() -> bool:
    """Verify pip is actually usable, not just importable. `find_spec` can
    return a spec even when pip's install is partially corrupted, so we
    also invoke `python -m pip --version` and check it succeeds."""
    if importlib.util.find_spec("pip") is None:
        return False
    check = subprocess.run(
        [sys.executable, "-m", "pip", "--version"],
        capture_output=True, text=True
    )
    return check.returncode == 0


def _clean_stale_pip_artifacts():
    """Best-effort self-heal: remove leftover '~xxx' folders in site-packages.
    Windows renames a locked file/folder to '~name' when it can't delete it
    outright (e.g. an interrupted pip self-upgrade), and pip logs these as
    'Ignoring invalid distribution ~ip/~ywin32/...'. Leftover ones can block
    a later ensurepip/pip repair, so clear them before repairing."""
    site_packages = Path(sysconfig.get_paths().get("purelib", ""))
    if not site_packages.exists():
        return
    stale = [p for p in site_packages.iterdir() if p.name.startswith("~")]
    if not stale:
        return
    print(f"  Found {len(stale)} stale artifact(s) in {site_packages}:")
    for p in stale:
        try:
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()
            print(f"    removed {p.name}")
        except OSError as e:
            print(f"    could not remove {p.name}: {e}")


# Step 1: verify pip first (cheap, no side effects) before attempting any repair/install.
print("\nVerifying pip...")
if _pip_is_functional():
    print("[ok] pip module found and functional")
else:
    print("[!!] pip not found or not functional for this interpreter.")

    # Step 2: self-heal — clear known-bad leftovers, then repair with ensurepip.
    _clean_stale_pip_artifacts()

    print("     Repairing with: python -m ensurepip --upgrade --user")
    result = subprocess.run(
        [sys.executable, "-m", "ensurepip", "--upgrade", "--user"],
        capture_output=True, text=True
    )
    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0 or not _pip_is_functional():
        # Common cause: this Python lives in a shared/admin-owned folder (e.g. c:\PythonXYZ)
        # and the current user can't write to its Scripts folder (WinError 5 Access is
        # denied). --user installs pip into the per-user site instead, which usually
        # avoids that. If it still fails, fall back to a system-wide attempt in case the
        # opposite is true (e.g. corporate policy blocks per-user installs).
        print("     --user repair had issues; retrying without --user...")
        result = subprocess.run(
            [sys.executable, "-m", "ensurepip", "--upgrade"],
            capture_output=True, text=True
        )
        print(result.stdout)
        print(result.stderr)

    if not _pip_is_functional():
        raise SystemExit(
            "Could not repair pip for this interpreter automatically.\n"
            "This is usually a permissions problem (WinError 5 Access is denied) writing to "
            f"'{Path(sys.executable).parent / 'Scripts'}', or a corrupted pip install. Try:\n"
            "  1) Close any other running Python/pip processes, then re-run this cell.\n"
            "  2) Run VS Code as the user who owns this Python install (or with elevated rights).\n"
            "  3) Reinstall Python from python.org (make sure 'pip' is checked) and restart the kernel.\n"
            "  4) Or switch the notebook kernel to a Python you fully own (e.g. a venv or per-user install)."
        )
    print("[ok] pip repaired.")

# Step 3: install the workshop's Python dependencies now that pip is confirmed working.
requirements_path = repo_dir / "tools" / "sql_mcp_server" / "requirements.txt"
cmd = f'"{sys.executable}" -m pip install --quiet -r "{requirements_path}" ipykernel'
result = run(cmd, check=False)
if result.returncode != 0:
    print("\n  Standard pip install failed; retrying with --user...")
    result = run(cmd.replace("pip install", "pip install --user"), check=False)
if result.returncode != 0:
    raise SystemExit(
        "pip install failed for this kernel's Python interpreter. "
        "Check network/proxy settings, then re-run this cell. "
        "See the '### Troubleshooting bootstrap issues' notes below for more checks."
    )
print("[ok] Packages installed (pyodbc, mcp, ipykernel) for the current notebook kernel.")


Python executable: /Users/scwong/Documents/projects/ms-workshop-build-ai-app/MS-LAB513-Student/.venv/bin/python
Python version   : 3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ]

Verifying pip...
[ok] pip module found and functional
$ "/Users/scwong/Documents/projects/ms-workshop-build-ai-app/MS-LAB513-Student/.venv/bin/python" -m pip install --quiet -r "/Users/scwong/Documents/projects/ms-workshop-build-ai-app/MS-LAB513-Student/tools/sql_mcp_server/requirements.txt" ipykernel
[ok] Packages installed (pyodbc, mcp, ipykernel) for the current notebook kernel.


In [7]:
# 0.1) Set DO_EXECUTE to True to actually run commands
# 0.2) Login to Azure and set the workshop subscription
DO_EXECUTE = True
if DO_EXECUTE:
    run(f"az login --tenant {TENANT_ID}")
    run(f"az account set --subscription {SUBSCRIPTION_ID}")
    run("az account show -o table")
else:
    print("Preview:")
    print(f"  az login --tenant {TENANT_ID}")
    print(f"  az account set --subscription {SUBSCRIPTION_ID}")


$ az login --tenant 506e2450-c16c-43fd-a391-6a014bf4af13
[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "506e2450-c16c-43fd-a391-6a014bf4af13",
    "id": "aae170e4-5b90-4bac-baf4-7aed0ebd6e4e",
    "isDefault": true,
    "managedByTenants": [],
    "name": "SRKK CAIP LAB",
    "state": "Enabled",
    "tenantId": "506e2450-c16c-43fd-a391-6a014bf4af13",
    "user": {
      "name": "student073@srkkcaiplab.onmicrosoft.com",
      "type": "user"
    }
  }
]


$ az account set --subscription aae170e4-5b90-4bac-baf4-7aed0ebd6e4e
$ az account show -o table
EnvironmentName    HomeTenantId                          IsDefault    Name           State    TenantId
-----------------  ------------------------------------  -----------  -------------  -------  ------------------------------------
AzureCloud         506e2450-c16c-43fd-a391-6a014bf4af13  True         SRKK CAIP LAB  Enabled  506e2450-c16c-43fd-a391-6a014bf4af13



In [8]:
# 0.2) Verify your identity and that lab files are present
if DO_EXECUTE:
    run('az ad signed-in-user show --query "{name:displayName, upn:userPrincipalName}" -o table')

print("searchfaq.sql present:", (repo_dir / "sql" / "searchfaq.sql").exists())
print("MCP server present   :", (repo_dir / "tools" / "sql_mcp_server" / "server.py").exists())


$ az ad signed-in-user show --query "{name:displayName, upn:userPrincipalName}" -o table
Name                Upn
------------------  --------------------------------------
LAB513 Student 073  student073@srkkcaiplab.onmicrosoft.com

searchfaq.sql present: True
MCP server present   : True


In [10]:
# 0.3) SQL helpers (Azure AD token auth via the ODBC Driver for SQL Server)
#      Requires: pip install pyodbc  and  'ODBC Driver 18 (or 17) for SQL Server' installed.
SQL_COPT_SS_ACCESS_TOKEN = 1256


def _pick_sql_driver():
    import pyodbc
    installed = [d for d in pyodbc.drivers() if "for SQL Server" in d]
    for preferred in ("ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server"):
        if preferred in installed:
            return preferred
    if installed:
        return installed[-1]
    raise RuntimeError(
        "No 'ODBC Driver for SQL Server' found. Install ODBC Driver 18: "
        "https://aka.ms/downloadmsodbcsql"
    )


def _sql_connect():
    import pyodbc
    driver = _pick_sql_driver()
    token = get_access_token("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    conn_str = (
        f"Driver={{{driver}}};"
        f"Server=tcp:{SQL_SERVER},1433;Database={SQL_DATABASE};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})


def run_sql(sql: str, params=None, fetch: bool = True):
    """Run a single SQL statement. Returns list[dict] for result sets."""
    conn = _sql_connect()
    try:
        cur = conn.cursor()
        cur.execute(sql, params or [])
        if fetch and cur.description:
            cols = [c[0] for c in cur.description]
            rows = [dict(zip(cols, r)) for r in cur.fetchall()]
            conn.commit()
            return rows
        conn.commit()
        return []
    finally:
        conn.close()


def run_sql_script(path: Path):
    """Run a .sql file that uses GO batch separators."""
    text = Path(path).read_text(encoding="utf-8")
    batches = []
    current = []
    for line in text.splitlines():
        if line.strip().upper() == "GO":
            batches.append("\n".join(current))
            current = []
        else:
            current.append(line)
    if current:
        batches.append("\n".join(current))
    conn = _sql_connect()
    try:
        cur = conn.cursor()
        for batch in batches:
            if batch.strip():
                cur.execute(batch)
        conn.commit()
    finally:
        conn.close()
    print(f"Executed {sum(1 for b in batches if b.strip())} batch(es) from {Path(path).name}")


print("SQL helpers ready. Using driver:", _pick_sql_driver())


SQL helpers ready. Using driver: ODBC Driver 18 for SQL Server


In [15]:
print(f"{OPENAI_ENDPOINT}/openai/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={CHAT_API_VERSION}")

https://ai-q4s3xh537i3xg.openai.azure.com/openai/deployments/o4-mini/chat/completions?api-version=2025-01-01-preview


In [ ]:
# 0.4) Verify the database is seeded and OpenAI responds
if DO_EXECUTE:
    count = run_sql("SELECT COUNT(*) AS FAQ_Count FROM dbo.FAQ_Content;")
    print("FAQ_Content rows:", count[0]["FAQ_Count"], "(expected 15+)")

    import urllib.request
    token = get_access_token("https://cognitiveservices.azure.com")
    body = json.dumps({"messages": [{"role": "user", "content": "Hello"}]}).encode()
    req = urllib.request.Request(
        f"{OPENAI_ENDPOINT}/openai/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={CHAT_API_VERSION}",
        data=body, method="POST",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read())
    print("OpenAI reply:", data["choices"][0]["message"]["content"])
else:
    print("Preview: run SELECT COUNT(*) FROM dbo.FAQ_Content and a test chat completion.")


FAQ_Content rows: 50 (expected 15+)
OpenAI reply: Hello! How can I help you today?


### ✅ Module 0 checklist
- [ ] `az account show` returns the workshop subscription
- [ ] SQL query works from this notebook
- [ ] `FAQ_Content` has 15+ records
- [ ] OpenAI `o4-mini` responds to a test message


---
## Module 1 — AI-Enhanced Querying with Azure SQL (20 min)

**Objective:** explore FAQ data, generate embeddings, and run FAQ retrieval with relevance scoring.


In [22]:
# 1.1) Explore FAQ content
if DO_EXECUTE:
    rows = run_sql(
        "SELECT TOP 10 id, category, question, LEFT(answer, 100) AS answer_preview "
        "FROM dbo.FAQ_Content ORDER BY category;"
    )
    for r in rows:
        print(f"[{r['id']}] ({r['category']}) {r['question']}")
else:
    print("Preview: SELECT TOP 10 ... FROM dbo.FAQ_Content ORDER BY category;")


[7] (Account) How do I set up a profile picture?
[6] (Account) What do I do if my account is locked?
[5] (Account) How do I link my Microsoft account?
[4] (Account) How do I change my username?
[3] (Account) How do I delete my account?
[2] (Account) How do I change my email address?
[1] (Account) How do I enable two-factor authentication?
[0] (Account) How do I reset my password?
[10] (Billing) Where can I find my invoices?
[9] (Billing) How do I cancel my subscription?


In [23]:
# 1.2) Create the embeddings table
CREATE_EMBEDDINGS = """
IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'FAQ_Embeddings')
BEGIN
    CREATE TABLE dbo.FAQ_Embeddings (
        id INT IDENTITY(1,1) PRIMARY KEY,
        content_id INT NOT NULL,
        embedding VARBINARY(MAX) NOT NULL,
        model_name NVARCHAR(100) NOT NULL DEFAULT 'text-embedding-ada-002',
        dimensions INT NOT NULL DEFAULT 1536,
        created_at DATETIME2 NOT NULL DEFAULT GETUTCDATE()
    );
END;
"""
if DO_EXECUTE:
    run_sql(CREATE_EMBEDDINGS, fetch=False)
    print("FAQ_Embeddings table ready.")
else:
    print("Preview: create dbo.FAQ_Embeddings if it does not exist.")


FAQ_Embeddings table ready.


In [24]:
# 1.3) Create the search procedure and function from sql/searchfaq.sql
if DO_EXECUTE:
    run_sql_script(repo_dir / "sql" / "searchfaq.sql")
else:
    print(f"Preview: run script {repo_dir / 'sql' / 'searchfaq.sql'}")


Executed 2 batch(es) from searchfaq.sql


In [25]:
# 1.4) Generate embeddings for each FAQ (Python port of the guide's script)
import urllib.request

if DO_EXECUTE:
    faqs = run_sql("SELECT id, question, answer FROM dbo.FAQ_Content;")
    aoai_token = get_access_token("https://cognitiveservices.azure.com")
    embed_url = (
        f"{OPENAI_ENDPOINT}/openai/deployments/{EMBEDDING_DEPLOYMENT}"
        f"/embeddings?api-version={EMBEDDING_API_VERSION}"
    )

    for row in faqs:
        text = f"{row['question']}\n{row['answer']}"
        body = json.dumps({"input": text}).encode()
        req = urllib.request.Request(
            embed_url, data=body, method="POST",
            headers={"Authorization": f"Bearer {aoai_token}", "Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req) as resp:
            vector = json.loads(resp.read())["data"][0]["embedding"]
        vec_bytes = json.dumps(vector, separators=(",", ":")).encode("utf-8")

        upsert = (
            "IF EXISTS (SELECT 1 FROM dbo.FAQ_Embeddings WHERE content_id=?) "
            "UPDATE dbo.FAQ_Embeddings SET embedding=?, model_name=?, dimensions=1536 WHERE content_id=? "
            "ELSE INSERT INTO dbo.FAQ_Embeddings (content_id, embedding, model_name, dimensions) "
            "VALUES (?,?,?,1536);"
        )
        run_sql(
            upsert,
            params=[row["id"], vec_bytes, EMBEDDING_DEPLOYMENT, row["id"], row["id"], vec_bytes, EMBEDDING_DEPLOYMENT],
            fetch=False,
        )
    print(f"Embeddings generated for {len(faqs)} FAQ rows.")
else:
    print("Preview: fetch FAQ rows, call the embeddings endpoint, upsert into FAQ_Embeddings.")


Embeddings generated for 50 FAQ rows.


In [26]:
# 1.5) Inspect embeddings and verify row counts match
if DO_EXECUTE:
    sample = run_sql(
        "SELECT TOP 5 id, content_id, DATALENGTH(embedding) AS embedding_bytes FROM dbo.FAQ_Embeddings;"
    )
    for r in sample:
        print(r)
    counts = run_sql(
        "SELECT 'FAQ_Content' AS TableName, COUNT(*) AS Rows FROM dbo.FAQ_Content "
        "UNION ALL SELECT 'FAQ_Embeddings', COUNT(*) FROM dbo.FAQ_Embeddings;"
    )
    for r in counts:
        print(r)
else:
    print("Preview: inspect DATALENGTH(embedding) and compare FAQ_Content vs FAQ_Embeddings counts.")


{'id': 1, 'content_id': 0, 'embedding_bytes': 19404}
{'id': 2, 'content_id': 1, 'embedding_bytes': 19429}
{'id': 3, 'content_id': 3, 'embedding_bytes': 19439}
{'id': 4, 'content_id': 4, 'embedding_bytes': 19397}
{'id': 5, 'content_id': 5, 'embedding_bytes': 19409}
{'TableName': 'FAQ_Content', 'Rows': 50}
{'TableName': 'FAQ_Embeddings', 'Rows': 50}


In [27]:
# 1.6) Run retrieval search and compare with raw keyword search
if DO_EXECUTE:
    for q in ["How do I reset my password?", "What payment methods do you accept?", "My account is locked"]:
        print(f"\n=== SearchFAQ: {q} ===")
        for r in run_sql("EXEC dbo.SearchFAQ @Question = ?;", params=[q]):
            print(f"  ({r['similarity_score']}) {r['question']}")

    print("\n=== Keyword match: '%password%' ===")
    for r in run_sql("SELECT question FROM dbo.FAQ_Content WHERE question LIKE '%password%';"):
        print("  ", r["question"])

    print("\n=== Retrieval scoring: 'I forgot my login credentials' ===")
    for r in run_sql("EXEC dbo.SearchFAQ @Question = ?;", params=["I forgot my login credentials"]):
        print(f"  ({r['similarity_score']}) {r['question']}")
else:
    print("Preview: EXEC dbo.SearchFAQ for several questions and compare with LIKE keyword search.")



=== SearchFAQ: How do I reset my password? ===
  (17.0) How do I reset my password?
  (10.0) How do I change my email address?
  (10.0) How do I upgrade my plan?

=== SearchFAQ: What payment methods do you accept? ===
  (10.0) What data formats do you support?
  (9.0) How do I update my payment method?
  (9.0) What security certifications do you have?

=== SearchFAQ: My account is locked ===
  (17.0) What do I do if my account is locked?
  (10.0) How do I delete my account?
  (9.0) How do I link my Microsoft account?

=== Keyword match: '%password%' ===
   How do I reset my password?

=== Retrieval scoring: 'I forgot my login credentials' ===
  (5.0) How do I change my email address?
  (5.0) What do I do if my account is locked?
  (4.0) How do I link my Microsoft account?


### ✅ Module 1 checklist
- [ ] `FAQ_Content` returns 15+ rows
- [ ] Embeddings generated for FAQ rows
- [ ] `FAQ_Embeddings` count matches `FAQ_Content`
- [ ] `SearchFAQ` returns relevant results
- [ ] Retrieval scoring matches intent better than raw keyword search


---
## Module 2 — Copilot-Assisted SQL Development (15 min)

**Objective:** use GitHub Copilot Chat to generate, explain, and refine SQL queries, then **test** the result two ways — in this notebook, or in the VS Code SQL Server extension.

### Step 1 — Generate SQL with Copilot Chat
Open **Copilot Chat** (Ctrl+Alt+I) and try these prompts one by one:

1. *"Generate a SQL query that performs FAQ retrieval scoring on `FAQ_Content` using terms from the user question and returns the top 3 results."*
2. *"Explain this query step by step, especially the scoring calculation."*
3. *"Add comments, meaningful aliases, a similarity threshold, and format for readability."*
4. *"Create a stored procedure `SearchFAQv2` with `@Question` and `@TopK` parameters, including error handling."*

> 💡 Copilot Chat uses your **GitHub/Copilot** sign-in (not Azure), so any identity can generate SQL. You only need `dryrun01` to *run* the SQL against the database.

### Step 2 — Test the generated query
Pick **either** option below. Both run against your seeded `FAQ_Content` table.

#### 🅰️ Option A — Run it in this notebook (uses your `dryrun01` CLI login)
Just run the **next code cell**. It executes the Copilot-style scoring query with `run_sql()` and, for comparison, the validated `EXEC dbo.SearchFAQ`. Change `question` in that cell to try other inputs.

#### 🅱️ Option B — Run it in the VS Code SQL Server extension
1. Open the **SQL Server** extension (left sidebar) → **Add Connection** (skip if already connected):
   - Server: `<your-sql-server>.database.windows.net`
   - Database: `faq-ai-assistant-db`
   - Authentication: **Microsoft Entra ID** → sign in as your workshop user (e.g. `dryrun01@...`)
2. Right-click the connection → **New Query**.
3. Paste the query below and press **Ctrl+Shift+E** (or the green ▶ Execute button):

```sql
DECLARE @UserQuestion NVARCHAR(500) = N'How do I reset my password?';

WITH terms AS (
    SELECT DISTINCT LTRIM(RTRIM(value)) AS term
    FROM STRING_SPLIT(LOWER(@UserQuestion), ' ')
    WHERE LEN(LTRIM(RTRIM(value))) > 1
),
scored AS (
    SELECT
        fc.id, fc.category, fc.question, fc.answer,
        SUM(
            CASE WHEN LOWER(fc.question) LIKE '%' + t.term + '%' THEN 3 ELSE 0 END +
            CASE WHEN LOWER(fc.category) LIKE '%' + t.term + '%' THEN 2 ELSE 0 END +
            CASE WHEN LOWER(fc.answer)   LIKE '%' + t.term + '%' THEN 1 ELSE 0 END
        ) AS relevance_score
    FROM dbo.FAQ_Content fc
    CROSS JOIN terms t
    GROUP BY fc.id, fc.category, fc.question, fc.answer
)
SELECT TOP (3) id, category, question, relevance_score
FROM scored
WHERE relevance_score > 0
ORDER BY relevance_score DESC, id ASC;
```

> ⚠️ **Always verify AI-generated SQL before executing.** The lab's tested queries are your ground truth. If you get *Login failed*, reconnect the extension using **Microsoft Entra ID**.

### ✅ Module 2 checklist
- [ ] Copilot generates syntactically correct SQL
- [ ] Generated query references correct table/column names
- [ ] Query runs (Option A notebook **or** Option B extension) and returns the top 3
- [ ] Top result matches `EXEC dbo.SearchFAQ`
- [ ] You understand AI-generated vs validated SQL


In [28]:
# 2.1) Option A — test the Copilot-generated scoring query in this notebook
#      (change `question` to try different inputs)
question = "How do I reset my password?"

COPILOT_SCORING_SQL = """
DECLARE @UserQuestion NVARCHAR(500) = ?;
WITH terms AS (
    SELECT DISTINCT LTRIM(RTRIM(value)) AS term
    FROM STRING_SPLIT(LOWER(@UserQuestion), ' ')
    WHERE LEN(LTRIM(RTRIM(value))) > 1
),
scored AS (
    SELECT
        fc.id, fc.category, fc.question, fc.answer,
        SUM(
            CASE WHEN LOWER(fc.question) LIKE '%' + t.term + '%' THEN 3 ELSE 0 END +
            CASE WHEN LOWER(fc.category) LIKE '%' + t.term + '%' THEN 2 ELSE 0 END +
            CASE WHEN LOWER(fc.answer)   LIKE '%' + t.term + '%' THEN 1 ELSE 0 END
        ) AS relevance_score
    FROM dbo.FAQ_Content fc
    CROSS JOIN terms t
    GROUP BY fc.id, fc.category, fc.question, fc.answer
)
SELECT TOP (3) id, category, question, relevance_score
FROM scored
WHERE relevance_score > 0
ORDER BY relevance_score DESC, id ASC;
"""

if DO_EXECUTE:
    print(f"=== Copilot-generated scoring query: {question} ===")
    for r in run_sql(COPILOT_SCORING_SQL, params=[question]):
        print(f"  ({r['relevance_score']}) [{r['category']}] {r['question']}")

    print("\n=== Validated procedure (ground truth): EXEC dbo.SearchFAQ ===")
    for r in run_sql("EXEC dbo.SearchFAQ @Question = ?;", params=[question]):
        print(f"  ({r['similarity_score']}) [{r['category']}] {r['question']}")
else:
    print("Preview: run the Copilot scoring query and EXEC dbo.SearchFAQ, then compare the top result.")


=== Copilot-generated scoring query: How do I reset my password? ===
  (17) [Account] How do I reset my password?
  (10) [Account] How do I change my email address?
  (10) [Billing] How do I upgrade my plan?

=== Validated procedure (ground truth): EXEC dbo.SearchFAQ ===
  (17.0) [Account] How do I reset my password?
  (10.0) [Account] How do I change my email address?
  (10.0) [Billing] How do I upgrade my plan?


---
## Module 3 — RAG Implementation (25 min)

**Objective:** retrieve FAQ context, augment a grounded prompt, and generate AI responses that only use provided context.


In [ ]:
# 3.1) Retrieve relevant FAQ content and build grounding context
def build_context(question: str, top_k: int = 3) -> str:
    rows = run_sql("EXEC dbo.SearchFAQ @Question = ?, @TopK = ?;", params=[question, top_k])
    return "\n\n".join(f"Q: {r['question']}\nA: {r['answer']}" for r in rows)


if DO_EXECUTE:
    ctx = build_context("How do I get a refund?")
    print(ctx)
else:
    print("Preview: EXEC dbo.SearchFAQ then STRING_AGG the top matches into a grounding context.")


In [ ]:
# 3.2) Construct a grounded prompt and call o4-mini
SYSTEM_PROMPT = (
    "You are a FAQ assistant. Answer ONLY using the FAQ context below. "
    "If the answer is not in the context, say 'I don't have that information.' "
    "Do NOT make up information."
)


def ask_grounded(question: str) -> str:
    context = build_context(question)
    token = get_access_token("https://cognitiveservices.azure.com")
    # NOTE: o4-mini is a reasoning model — it only supports the default temperature (1),
    # so we do not send a temperature parameter (sending 0.3 returns HTTP 400).
    body = json.dumps({
        "messages": [
            {"role": "system", "content": f"{SYSTEM_PROMPT}\n\nFAQ CONTEXT:\n{context}"},
            {"role": "user", "content": question},
        ],
    }).encode()
    req = urllib.request.Request(
        f"{OPENAI_ENDPOINT}/openai/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={CHAT_API_VERSION}",
        data=body, method="POST",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())["choices"][0]["message"]["content"]


if DO_EXECUTE:
    print(ask_grounded("How do I get a refund?"))
else:
    print("Preview: send system(grounding+context) + user question to o4-mini (default temperature).")


In [ ]:
# 3.3) Test with unsupported questions — the assistant should decline
if DO_EXECUTE:
    for q in ["What's the weather today?", "Tell me about company history"]:
        print(f"Q: {q}\nA: {ask_grounded(q)}\n")
else:
    print("Preview: out-of-scope questions should return 'I don't have that information.'")


### ✅ Module 3 checklist
- [ ] Search returns relevant FAQ context
- [ ] Grounded prompt constrains the AI to FAQ content
- [ ] Responses reference only the provided context
- [ ] Unsupported questions get "I don't have that information"
- [ ] No hallucinations observed


---
## Module 4 — Foundry Agent Orchestration (25 min)

**Objective:** create a Microsoft Foundry Agent that uses MCP tool calls to retrieve FAQ answers.

The MCP server and dev tunnel must stay running, and the Agent is created in the **Foundry portal** — so the server/tunnel commands are shown for you to run in **separate terminals**.


In [ ]:
# 4.1) Pre-check for Module 4
print("MCP server present:", (repo_dir / "tools" / "sql_mcp_server" / "server.py").exists())
if DO_EXECUTE:
    run("python --version", check=False)      # need 3.10+
    run("devtunnel --version", check=False)    # need Dev Tunnels CLI
else:
    print("Preview: python --version (3.10+) and devtunnel --version.")


This module has three parts, run in **separate terminals** (keep them open) plus the Foundry portal. These steps are the **tested** sequence — follow them exactly to avoid the common "Connect is greyed out" problem.

### Terminal 1 — start the MCP server
```powershell
cd .\tools\sql_mcp_server
# use your local Python environment (has pyodbc + mcp)
pip install -r requirements.txt
$env:SQL_SERVER  = "<your-sql-server>.database.windows.net"
$env:SQL_DATABASE = "faq-ai-assistant-db"
$env:PORT = "5000"
python server.py
```
Wait for `Uvicorn running on http://0.0.0.0:5000`. **Keep this terminal running.**

> 🔑 **The server uses *your* Azure CLI login for SQL.** Make sure the terminal that runs `server.py` is signed in as your workshop user (`az account show` → `dryrun01@...`). The agent runs in the Foundry portal, but the actual SQL query happens on *this* machine using *this* terminal's `az login` token — so this identity must be the SQL user.

### Terminal 2 — expose it with a dev tunnel (⚠️ note `--protocol http`)
```powershell
devtunnel user login                                   # sign in (Microsoft/Entra) — required
devtunnel create faq-[alias] --allow-anonymous
devtunnel port create faq-[alias] -p 5000 --protocol http   # --protocol http is REQUIRED
devtunnel host faq-[alias]
```
From the output copy the **forwarding URL** (looks like `https://<name>-5000.<region>.devtunnels.ms`). Your **MCP endpoint** is that URL **+ `/mcp`**. Keep this terminal running.

> 🔑 **Why `--protocol http` matters:** without it, the tunnel applies its web‑forwarding/anti‑phishing gate, which blocks Foundry's server‑side connection probe and leaves the **Connect** button greyed out. Creating the port as `http` makes the endpoint reachable by Foundry.

### In the Microsoft Foundry portal
1. Open **https://ai.azure.com/** → **Sign in** → open the **`FAQ-Assistant-project`** project → **Let's go**.
2. Select **Build** → **Tools** (left menu) → **Tools** tab → **Connect a tool**.
3. On the **Custom** tab choose **Model Context Protocol (MCP)** → **Create**.
4. Configure the connection:
   - **Name:** `faq-tool`
   - **Remote MCP Server endpoint:** `<your-tunnel-url>/mcp`
   - **Authentication:** **Unauthenticated** (if you want to use Entra ID authentication instead, make sure your `devtunnel host` is showing it's ready to accept connections and the connection hasn't terminated)
5. Select **Connect** → then **Use in an agent**.
6. Agent name: `faq-orchestrator-agent` → **Create and open playground**.
7. In **Instructions**, paste:
   ```
   You are a support FAQ assistant.
   Use the available MCP tool to retrieve relevant FAQ content before answering.
   Answer using only the tool results when possible.
   If the tool does not return relevant information, say that you do not know.
   Do not invent policies, refunds, or details that are not in the FAQ content.
   ```
8. Confirm Foundry **discovers the tools** (`search_faq`, `list_faq_categories`), then **Save**.

### Step 4 — Test the agent in the playground (tested sequence)
With the server + tunnel running and the tool connected:

1. In the agent **Chat** panel, type a **supported** question and send it:
   ```
   How do I reset my password?
   ```
2. The agent decides to call the tool and Foundry shows an **`mcp approval request`** with the exact call, e.g.:
   ```
   search_faq({ "question": "how do i reset my password", "top_k": 3 })
   ```
   Click **✅ Approve**. *(See the note below on why this appears.)*
3. Watch **Terminal 1** — you should see the incoming `POST /mcp` and a `CallToolRequest` for `search_faq` (no errors).
4. The agent returns a **grounded answer** built from the FAQ rows (password‑reset steps).
5. Now test an **out‑of‑scope** question:
   ```
   Can I pay using cryptocurrency?
   ```
   The tool returns no relevant FAQ, so the agent should **decline gracefully** (e.g. *"I don't have information about that."*) instead of inventing an answer.

> 💡 **Why the "Approve" prompt appears (this is expected):** Foundry defaults to **human‑in‑the‑loop approval** for every MCP tool call. Before the agent actually calls `search_faq` on your server, Foundry pauses and asks you to **Approve/Deny** that specific invocation — a safety gate so an agent can't silently call external tools. This is *not* an error.
>
> **To make the demo hands‑free (optional):** in the **Tools** panel, click the **⋮** next to **faq-tool** → edit the tool → set **Require approval** to **Never** (a.k.a. "Do not require approval") → **Save**. The agent will then call the tool automatically with no prompt. For teaching, it's nice to leave approval **on** for the first run so students *see* the tool fire, then switch it off.

### ✅ Module 4 checklist
- [ ] MCP server running (`Uvicorn running on http://0.0.0.0:5000`) under your workshop identity (`az account show`)
- [ ] Dev tunnel hosting with the port created as `--protocol http`
- [ ] Foundry MCP tool connects (Authentication = **Unauthenticated**) and discovers the tools
- [ ] Supported question ("How do I reset my password?") → **Approve** the tool call → grounded answer
- [ ] `CallToolRequest` for `search_faq` appears in the server terminal with no errors
- [ ] Unsupported question ("Can I pay using cryptocurrency?") → grounded fallback ("I don't have information…")

### 🆘 If **Connect** is greyed out / fails
1. **Recreate the port with `--protocol http`** (the #1 cause):
   ```powershell
   devtunnel delete faq-tunnel
   devtunnel create faq-tunnel --allow-anonymous
   devtunnel port create faq-tunnel -p 5000 --protocol http
   devtunnel host faq-tunnel
   ```
2. Make sure **Authentication = Unauthenticated** (not Microsoft Entra — Entra requires an Audience and blocks Connect).
3. Open `<your-tunnel-url>/mcp` once in a browser and click **Continue** on the dev‑tunnel notice to activate the tunnel, then retry.
4. Confirm the endpoint ends in **`/mcp`** with no trailing space, and both terminals (server + tunnel) are still running.

### 🆘 If the tool call errors (agent says it can't get data)
1. **`[WinError 2]` / `az` not found in the server terminal** → the server must launch `az` via the shell on Windows. The workshop `server.py` already does this (`shell=True`); make sure you're running the repo's `server.py`, not an older copy.
2. **`Login failed` / token errors** → the *server terminal's* `az login` must be your SQL user. Run `az account show` in Terminal 1; if it's the wrong account, `az login --tenant <tenant>` there and restart the server.
3. **Tunnel URL changed** → each new `devtunnel host` can issue a new URL. If you recreated the tunnel, update the **Remote MCP Server endpoint** in the Foundry tool (`<new-url>/mcp`) and reconnect.

In [ ]:
# # 4.2) OPTIONAL: launch the MCP server from the notebook (non-blocking)
# #      Prefer a dedicated terminal so you can watch its logs. Uncomment to try.
# # if DO_EXECUTE:
# #     env = os.environ.copy()
# #     env["SQL_SERVER"] = SQL_SERVER
# #     env["SQL_DATABASE"] = SQL_DATABASE
# #     env["SQL_TOPK"] = "3"
# #     server_dir = repo_dir / "tools" / "sql_mcp_server"
# #     run(f'pip install -r "{server_dir / "requirements.txt"}"')
# #     proc = subprocess.Popen(["python", "server.py"], cwd=str(server_dir), env=env)
# #     print("MCP server PID:", proc.pid, "(stop with proc.terminate())")
# print("See the terminal commands above — running the server in its own terminal is recommended.")


---
## Module 5 — Microsoft Fabric Integration (20 min)

**Objective:** mirror Azure SQL data into Fabric, then analyze it with a semantic model and Power BI report. Done in the **Fabric portal** ([app.fabric.microsoft.com](https://app.fabric.microsoft.com)). These are the **tested** steps — follow them in order.

> ⚠️ **Two prerequisites are handled by the workshop Bicep** (so you don't have to configure them), but know what they are in case something fails:
> 1. **Fabric capacity** — the self‑service Fabric trial is **blocked by tenant policy** (you'll see *"Sorry, we are unable to start trial at this time"*). Your instructor provisions a **Fabric `F2` capacity** named `faqfabric<suffix>`.
> 2. **SQL system‑assigned managed identity (SAMI)** — the SQL *logical server* must have its **SAMI enabled and primary**, or mirroring fails with *"turn on the system-assigned managed identity … primary identity"*. Bicep sets `identity: { type: 'SystemAssigned' }` on the server.

### Step 1 — Create a workspace and assign the Fabric capacity
1. **Workspaces → + New workspace** → name `LAB513-<your-alias>-FAQ` → **Apply**.
2. **Workspace settings** (⚙ / `...` → Workspace settings) → **Workspace type** → **Edit**.
3. Choose the **Fabric capacity** option → in **Details** pick **`faqfabric<suffix>`** (region *Southeast Asia*).
4. **Semantic model storage format:** leave **Small** (default; Large is only for >10 GB models).
5. Click **Select**. The workspace type changes from *Power BI Pro* to your Fabric capacity.

> Without a capacity, mirroring can't run. Ignore the **"Start Fabric trial"** button — it's policy‑blocked and not needed.

### Step 2 — Create the mirrored Azure SQL Database
1. In the workspace: **+ New item → Mirrored Azure SQL Database**.
2. **New source:**
   - **Server:** `<your-sql-server>.database.windows.net`
   - **Database:** `faq-ai-assistant-db`
3. **Connection credentials:**
   - **Connection:** *Create new connection*
   - **Data gateway:** *(none)*
   - **Authentication kind:** **Organizational account** ← *not* Service principal
   - Click **Sign in / it shows "You are currently signed in as …"** → confirm it's your workshop user (`dryrun01@...`, the SQL Entra admin)
   - **Privacy level:** *Organizational* (or *None*)
4. Click **Connect**.

> 🆘 **If you see *"turn on the system-assigned managed identity …"***: this is a SQL **server** setting your instructor enables (it's set automatically for fresh Bicep deployments). Notify your instructor, wait 1–2 min, **close the dialog**, and start *New mirrored database* again.
>
> 🆘 **If the connection screen asks for Tenant ID / client ID / key**: you accidentally selected **Service principal** — switch **Authentication kind** back to **Organizational account**.

### Step 3 — Choose data and start replication
1. On **Choose data**, select ✅ **`dbo.FAQ_Content`** (you may also include `dbo.FAQ_Embeddings`).
2. Optionally leave **"Automatically mirror future tables"** checked.
3. Click **Connect**. Fabric creates the mirrored database and begins replicating.
4. **Wait for sync (1–3 min):** each table's **status** moves to **Running / Replicated** and shows a row count (`FAQ_Content` = 15).

### Step 4 — Query the SQL analytics endpoint
1. Open the mirrored database → switch to the **SQL analytics endpoint** (selector at the top of the item).
2. **New SQL query** and run:
   ```sql
   SELECT category, COUNT(*) AS faq_count
   FROM dbo.FAQ_Content
   GROUP BY category
   ORDER BY faq_count DESC;
   ```
   You get the FAQ categories with counts — the same result as Module 1, now served from OneLake.

### Step 5 — Create a semantic model (Direct Lake)
1. From the SQL analytics endpoint toolbar → **New semantic model** (or **Reporting → New semantic model**).
2. Name it `FAQ_Model`, select the **`FAQ_Content`** table → **Confirm**.
   - This is a **Direct Lake** model — it reads mirrored data directly from OneLake (no import/refresh), which is the point of mirroring.

### Step 6 — Build the Power BI report (category vs count)
1. From the semantic model → **Create report** (or **New report**).
2. In **Visualizations**, choose **Clustered column chart** (or **Clustered bar chart**).
3. Click the **Build visual** icon (bar‑chart icon, left of "Format visual") to show the field wells, then:
   - **X‑axis** (column) / **Y‑axis** (bar) → drag **`category`**
   - **Y‑axis** (column) / **X‑axis** (bar) → drag **`id`** → open its dropdown → **Count**
   > ⚠️ **Common mistake:** putting **`id`** on the axis makes one bar per row (15 bars of 1). Put **`category`** on the axis and **Count of `id`** as the value to get one bar per category.
4. **Save** (top‑right) → name `FAQ Category Report` → into the `LAB513-<your-alias>-FAQ` workspace.

### Step 7 — Review data lineage
- In the workspace, switch to **Lineage view** (view toggle, top‑right) and confirm the chain:
  **Azure SQL → Mirrored DB → SQL analytics endpoint → Semantic model → Report.**

### ✅ Module 5 checklist
- [ ] Workspace created **and assigned to the Fabric capacity** (`faqfabric<suffix>`)
- [ ] SQL server has **system-assigned managed identity** (primary)
- [ ] Mirrored database connected with **Organizational account** and shows **Running/Replicated** (15 rows)
- [ ] SQL analytics endpoint returns FAQ data grouped by category
- [ ] **Direct Lake** semantic model created on `FAQ_Content`
- [ ] Report shows **count of FAQs per category** (category on axis, Count of id as value)
- [ ] Lineage shows the full data-flow chain

### 🆘 Fabric troubleshooting
| Issue | Fix |
|-------|-----|
| "Unable to start trial" | Expected — trials are policy-blocked. Use the instructor's **Fabric capacity** (Workspace settings → Workspace type → Fabric capacity). |
| Workspace has no capacity to run mirroring | Assign the workspace to `faqfabric<suffix>` (Workspace type = Fabric capacity). |
| *"turn on the system-assigned managed identity … primary identity"* | SQL server setting your instructor enables (set automatically by Bicep for fresh deployments) — notify your instructor, then retry. |
| Connection asks for Tenant ID / client ID / key | You selected **Service principal** — switch **Authentication kind** to **Organizational account**. |
| Mirror **Connect** fails / can't reach SQL | Ensure SQL public access is on and the **`AllowAzureServices`** (`0.0.0.0`) firewall rule exists (Bicep sets it). |
| "Login failed" mirroring | Use **Organizational account** as your workshop user (the SQL Entra admin). |
| Report shows one bar per row | Put **`category`** on the axis and **Count of `id`** as the value (not `id` on the axis). |


---
## Module 6 — SQL MCP Server with Data API Builder (20 min)

**Objective:** expose Azure SQL as an MCP-compatible tool using Data API Builder (DAB), queryable from VS Code Copilot Chat.

> Stop the Python MCP server from Module 4 before starting DAB.


In [ ]:
# 6.1) Pre-check + install Data API Builder
if DO_EXECUTE:
    run("dotnet --version")  # need 8.0+
    run("dotnet tool install --global microsoft.dataapibuilder", check=False)
    run("dab --version")
else:
    print("Preview: dotnet --version (8.0+), then dotnet tool install --global microsoft.dataapibuilder.")


In [ ]:
# 6.2) Initialize the DAB config and add the FAQ entity (read-only)
lab_dir = repo_dir / "sql-mcp-lab"
conn_str = (
    f"Server=tcp:{SQL_SERVER},1433;Database={SQL_DATABASE};"
    "Authentication=Active Directory Default;Encrypt=True;TrustServerCertificate=False;"
)
if DO_EXECUTE:
    lab_dir.mkdir(exist_ok=True)
    # pushd/popd so dab writes dab-config.json INTO sql-mcp-lab (dab uses the cwd).
    run(f'pushd "{lab_dir}" && dab init --database-type "mssql" --connection-string "{conn_str}" --host-mode "Development" && popd')
    run(f'pushd "{lab_dir}" && dab add FAQ_Content --source "dbo.FAQ_Content" --permissions "anonymous:read" && popd')
    print("DAB config created in:", lab_dir)
else:
    print("Preview: create sql-mcp-lab/, dab init, dab add FAQ_Content (anonymous:read).")
print("Config lives in:", lab_dir / "dab-config.json")


### Wire DAB into VS Code as an MCP server (tested)

> ⚠️ **Identity:** DAB uses `Authentication=Active Directory Default`, which picks up your **`az login`** token. Make sure the CLI is your **workshop user / SQL admin** (`az account show` → `dryrun01@...`) — a Contributor-only account that isn't a SQL user will fail with *Login failed*.
>
> ⚠️ **Flag change:** in DAB **2.0.9** the stdio MCP flag is **`dab start --mcp-stdio`** (the older `--mcp` is not recognized). For VS Code, **you do not run this yourself** — VS Code launches DAB over stdio using `mcp.json`.

1. Create **`.vscode/mcp.json`** in the repo root:
   ```json
   {
     "servers": {
       "sql-faq": {
         "type": "stdio",
         "command": "dab",
         "args": ["start", "--mcp-stdio"],
         "cwd": "${workspaceFolder}/sql-mcp-lab"
       }
     }
   }
   ```
2. Open `.vscode/mcp.json` — VS Code shows a **Start** codelens above the `sql-faq` server. Click **Start** (status should go to *Running*).
3. Open **Copilot Chat**, switch to **Agent** mode, and confirm the **sql-faq** tools appear in the tools picker (🔧). Enable them.
4. Ask natural-language questions that hit the tool:
   - *"Using the sql-faq tools, what FAQ categories are available?"*
   - *"Show me all Account FAQs."*
   - *"How do I reset my password?"*


- [ ] Data is read-only (cannot modify via DAB)

### ✅ Module 6 checklist- [ ] Natural-language queries return real FAQ data

- [ ] DAB installed and version confirmed (`dab --version`)- [ ] VS Code Copilot Chat (Agent mode) detects the **sql-faq** tools

- [ ] `az account show` is your SQL user (`dryrun01@...`)- [ ] `.vscode/mcp.json` uses `dab start --mcp-stdio` and starts without errors
- [ ] `dab-config.json` created in `sql-mcp-lab` with the `FAQ_Content` entity (anonymous:read)

## [Optional] Module 7 — FAQ Assistant Chat Interface (15 min)

**Objective:** run a self-contained web chat UI over the Module 3 RAG pipeline —
type a question → retrieve FAQ context from Azure SQL → grounded o4-mini answer,
with the matched **sources** shown. No Copilot or Foundry required; it uses only
`pyodbc`, your `az login`, and the Python standard library.

The app lives at [`faq_agent_app.py`](faq_agent_app.py). Run it in a **separate terminal**
so it keeps serving while you chat:

```powershell
cd .\workshop
python faq_agent_app.py

In [ ]:
# 7.1) OPTIONAL: launch the FAQ Assistant web app from the notebook (non-blocking)
#      A dedicated terminal is recommended so you can watch logs and stop it with Ctrl+C:
#          cd workshop
#          python faq_agent_app.py
#      Uncomment below to launch it from here instead.
# import subprocess, sys, webbrowser
# app_path = workshop_dir / "faq_agent_app.py"
# proc = subprocess.Popen([sys.executable, str(app_path)], cwd=str(workshop_dir))
# print("FAQ Assistant PID:", proc.pid, "-> http://localhost:8000  (stop with proc.terminate())")
# webbrowser.open("http://localhost:8000")
print("Run in a terminal:  cd workshop  then  python faq_agent_app.py   ->  http://localhost:8000")

---
## 🎉 Workshop Complete!

You built:

| ✅ | Achievement |
|----|------------|
| 1 | Retrieval-based semantic search in Azure SQL |
| 2 | AI-assisted SQL development with Copilot |
| 3 | RAG pipeline with grounded AI responses |
| 4 | Foundry Agent orchestrating FAQ workflow via MCP |
| 5 | Real-time Fabric mirroring with Power BI analytics |
| 6 | Database-as-MCP-tool via Data API Builder |

### 🧹 Cleanup (only if your instructor confirms)
```powershell
az group delete --name "rg-<your-env-name>" --yes --no-wait
```

### 🆘 Quick Troubleshooting

| Issue | Solution |
|-------|----------|
| "Login failed" on SQL | Use Entra ID auth; run `az login` to refresh. |
| `pyodbc` / driver error | `pip install pyodbc` and install *ODBC Driver 18 for SQL Server*. |
| OpenAI "Resource not found" | Check API version is `2025-01-01-preview`. |
| Foundry **Connect** greyed out | Recreate the tunnel port with `--protocol http`; set tool auth to **Unauthenticated**; endpoint must end in `/mcp`. |
| Foundry asks to **Approve** each tool call | Expected (human-in-the-loop). Click **Approve**, or set the tool's **Require approval → Never** for hands-free runs. |
| Agent tool call fails with `[WinError 2]` | The server must call `az` via the shell on Windows — run the repo's `server.py` (already uses `shell=True`). |
| Agent can't get data / `Login failed` in server log | The **server terminal's** `az login` must be your SQL user — check with `az account show`, re-login, and restart the server. |
| Dev tunnel disconnects | Create a new tunnel, then update the Foundry agent's MCP endpoint (`<new-url>/mcp`). |
| Fabric "Cannot connect" | Ensure Entra auth; check the firewall allows Fabric. |
| DAB `Option 'mcp' is unknown` | DAB 2.0.9 uses **`dab start --mcp-stdio`** (not `--mcp`). |
| DAB "Config not provided / doesn't exist" | Run from `sql-mcp-lab` or pass `-c "<repo>\sql-mcp-lab\dab-config.json"`. |
| DAB "Authentication failed" / Login failed | The CLI must be your SQL user (`az account show` → `dryrun01@...`); `az login` and restart. |
| Copilot doesn't see MCP tool | Open `.vscode/mcp.json`, click **Start**, and enable the tool in Copilot Chat **Agent** mode. |
